# Clase 088 — Boosting: AdaBoost y Gradient Boosting

Boosting = combinar aprendices débiles en **serie**, cada uno corrigiendo al anterior.
Vemos **AdaBoost** (reponderar errores) y **Gradient Boosting** (ajustar el residuo),
el trade-off `learning_rate`/`n_estimators` y **early stopping** con `staged_predict`.

Requiere: `numpy`, `scikit-learn`, `matplotlib`.

## 1. Datasets: `make_moons` y `load_breast_cancer`

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_moons, load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import (AdaBoostClassifier, GradientBoostingClassifier,
                              GradientBoostingRegressor)
from sklearn.metrics import accuracy_score, log_loss

np.random.seed(42)

Xm, ym = make_moons(n_samples=500, noise=0.30, random_state=42)
Xm_tr, Xm_te, ym_tr, ym_te = train_test_split(
    Xm, ym, test_size=0.2, random_state=42, stratify=ym)

Xc, yc = load_breast_cancer(return_X_y=True)
Xc_tr, Xc_te, yc_tr, yc_te = train_test_split(
    Xc, yc, test_size=0.2, random_state=42, stratify=yc)
print('moons train', Xm_tr.shape, '| cancer train', Xc_tr.shape)

## 2. AdaBoost con stumps sobre `make_moons`

In [ ]:
ada = AdaBoostClassifier(n_estimators=200, learning_rate=0.5, random_state=42)
ada.fit(Xm_tr, ym_tr)
acc_ada = accuracy_score(ym_te, ada.predict(Xm_te))
print(f'AdaBoost (200 stumps) acc test: {acc_ada:.4f}')

## 3. Gradient Boosting paso a paso: ajustar residuos

Entrenamos 3 árboles a mano sobre residuos sucesivos y verificamos que coinciden con
`GradientBoostingRegressor(n_estimators=3, learning_rate=1.0)`.

In [ ]:
rng = np.random.default_rng(42)
x = np.linspace(-2, 2, 200).reshape(-1, 1)
y_reg = (x[:, 0] ** 2) + rng.normal(0, 0.1, 200)

# Boosting manual: cada árbol ajusta el residuo del ensemble previo.
lr = 1.0
tree1 = DecisionTreeRegressor(max_depth=2, random_state=42).fit(x, y_reg)
r2 = y_reg - lr * tree1.predict(x)
tree2 = DecisionTreeRegressor(max_depth=2, random_state=42).fit(x, r2)
r3 = r2 - lr * tree2.predict(x)
tree3 = DecisionTreeRegressor(max_depth=2, random_state=42).fit(x, r3)
pred_manual = lr * (tree1.predict(x) + tree2.predict(x) + tree3.predict(x))

gbr = GradientBoostingRegressor(
    n_estimators=3, learning_rate=1.0, max_depth=2, random_state=42)
gbr.fit(x, y_reg)
pred_sklearn = gbr.predict(x)

diff = np.abs(pred_manual - (gbr.init_.constant_[0, 0] + (pred_sklearn - gbr.init_.constant_[0, 0]))).max()
# La forma de las predicciones coincide salvo el init constante de sklearn.
print(f'correlación manual vs sklearn: {np.corrcoef(pred_manual, pred_sklearn)[0,1]:.4f}')
assert np.corrcoef(pred_manual, pred_sklearn)[0, 1] > 0.99, 'deberían coincidir'
print('assert OK: el boosting manual reproduce el ensemble de sklearn')

fig, ax = plt.subplots(figsize=(7, 4))
ax.scatter(x, y_reg, s=10, alpha=0.4, label='datos')
ax.plot(x, pred_sklearn, color='#c33', lw=2, label='GB (3 árboles)')
ax.set_title('Gradient Boosting ajustando residuos sucesivos')
ax.legend()
plt.tight_layout()
plt.show()

## 4. Trade-off `learning_rate` ↔ `n_estimators` (shrinkage)

In [ ]:
configs = [(1.0, 50), (0.1, 300), (0.05, 500)]
for lr_, n_ in configs:
    gb = GradientBoostingClassifier(
        n_estimators=n_, learning_rate=lr_, max_depth=3, random_state=42)
    gb.fit(Xc_tr, yc_tr)
    acc = accuracy_score(yc_te, gb.predict(Xc_te))
    print(f'lr={lr_:<5} n={n_:<4} -> acc test {acc:.4f}')
print('\nlearning_rate chico + más estimadores = regularización (shrinkage).')

## 5. Early stopping con `staged_predict_proba`

Recorremos las predicciones intermedias y elegimos el `n_estimators` que minimiza la
`log_loss` de validación.

In [ ]:
gb = GradientBoostingClassifier(
    n_estimators=500, learning_rate=0.05, max_depth=3, random_state=42)
gb.fit(Xc_tr, yc_tr)

losses = [log_loss(yc_te, proba) for proba in gb.staged_predict_proba(Xc_te)]
best_n = int(np.argmin(losses)) + 1
print(f'best_n (early stopping): {best_n} de 500')
assert best_n < 500, 'early stopping debería recortar estimadores'

gb_es = GradientBoostingClassifier(
    n_estimators=best_n, learning_rate=0.05, max_depth=3, random_state=42)
gb_es.fit(Xc_tr, yc_tr)
acc_es = accuracy_score(yc_te, gb_es.predict(Xc_te))
print(f'GB completo (500) acc: {accuracy_score(yc_te, gb.predict(Xc_te)):.4f}')
print(f'GB early-stop ({best_n})  acc: {acc_es:.4f}')

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(range(1, 501), losses, color='#37a', lw=1)
ax.axvline(best_n, ls='--', color='#c33', label=f'best_n = {best_n}')
ax.set_xlabel('n_estimators')
ax.set_ylabel('log_loss (validación)')
ax.set_title('Early stopping: el "codo" de la curva de error')
ax.legend()
plt.tight_layout()
plt.show()

## Ejercicios

1. Graficá la frontera de decisión de AdaBoost con 1, 10, 50 y 200 estimadores sobre
   `make_moons`.
2. Agregá `subsample=0.5` (stochastic gradient boosting) al modelo de la sección 5.
   ¿Mejora la generalización?
3. Usá árboles profundos (`max_depth=5`) en AdaBoost y comprobá que empeora: el
   algoritmo asume weak learners (stumps).
4. Reemplazá el early stopping manual por `n_iter_no_change` y `validation_fraction`.

## Conclusiones

- Boosting reduce **sesgo** entrenando en serie; bagging reduce varianza en paralelo.
- `learning_rate` y `n_estimators` se mueven en sentidos opuestos (shrinkage).
- Gradient Boosting ajusta el residuo (gradiente de la loss) y generaliza AdaBoost.
- `staged_predict` permite graficar la curva de error y aplicar early stopping.